In [ ]:
import os
import PIL as pil
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torchvision.utils import make_grid
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32
NUM_WORKERS = 8

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base="data"):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]


get_folder_structure()

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir a escala de grises
        return F.to_pil_image(combined_image)

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        #  transforms.Grayscale(),
        BenGrahamPreprocessing(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ]
)

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root="data/train/", transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
testset = torchvision.datasets.ImageFolder(root="data/test/", transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

classes = ("0-noDR", "1-mild", "2-moderate", "3-severe", "4-proliferativeDR")

In [ ]:
print("Follwing classes are there : \n", trainset.classes)

In [ ]:
def display_img(img, label):
    print(f"Label : {trainset.classes[label]}")
    plt.imshow(img.permute(1, 2, 0), cmap="gray")


# display the first image in the dataset
display_img(*trainset[0])

In [ ]:
# Cargar un modelo preentrenado
num_classes = 5  # Cambia a tu número de clases si es diferente
model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)

for param in model.features.parameters():
    param.requires_grad = False

# Modificar el clasificador para clasificación binaria
num_features = model.classifier[6].in_features
model.classifier[6] = torch.nn.Linear(num_features, num_classes)

# Ajustar la capa final para tu número de clases (puedes cambiar este ejemplo si hay más de 2 clases)
# Llevar el modelo a GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# criterion = torch.nn.NLLLoss(reduction="sum")  # Usa NLLLoss para clasificación binaria
criterion = torch.nn.CrossEntropyLoss()  # Usa CrossEntropy para clasificación múltiple
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)
# lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',factor=0.5, patience=20)

In [ ]:
torch.set_num_threads(os.cpu_count())  # Usa todos los cores

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    """
    Entrena el modelo.

    Args:
        model: El modelo a entrenar.
        criterion: La función de pérdida.
        optimizer: El optimizador.
        num_epochs: El número de épocas de entrenamiento.

    Returns:
        El modelo entrenado.
    """
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}/{num_epochs - 1}")
        print("-" * 10)

        # Iterar sobre las fases de entrenamiento y validación
        for phase in ["train"]:  # , 'val']:
            if phase == "train":
                model.train()  # Poner el modelo en modo de entrenamiento
            else:
                model.eval()  # Poner el modelo en modo de evaluación

            running_loss = 0.0
            running_corrects = 0

            # Iterar sobre los datos en batches
            for inputs, labels in tqdm(trainloader):  # dataloaders[phase]:
                inputs = inputs.to(device)  # Mover los datos al dispositivo
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Habilitar el cálculo de gradientes solo en la fase de entrenamiento
                with torch.set_grad_enabled(phase == "train"):
                    # Forward pass: calcular las predicciones
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)  # Obtener la clase predicha
                    loss = criterion(outputs, labels)  # Calcular la pérdida

                    # Backward pass + optimizar solo en la fase de entrenamiento
                    if phase == "train":
                        loss.backward()  # Calcular los gradientes
                        optimizer.step()  # Actualizar los pesos del modelo

                # Calcular estadísticas
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Calcular la pérdida y la precisión para la época
            epoch_loss = running_loss / len(trainset)  # dataset_sizes[phase]
            epoch_acc = running_corrects.double() / len(
                trainset
            )  # dataset_sizes[phase]

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        print()

    print("Entrenamiento completado")
    return model

In [ ]:
train_model(model, criterion, optimizer, num_epochs=10)

In [ ]:
num_epochs = 10  # Ajusta como sea necesario

for epoch in range(num_epochs):
    # model.train()  # Establece el modo de entrenamiento
    running_loss = 0.0

    for images, labels in tqdm(trainloader):
        images, labels = (
            images.to(device),
            labels.to(device),
        )  # Mueve a GPU si es necesario

        optimizer.zero_grad()  # Limpia los gradientes

        outputs = model(images)  # Propagación hacia adelante

        loss = criterion(outputs, labels)  # Calcular la pérdida
        loss.backward()  # Propagación hacia atrás
        optimizer.step()  # Actualizar pesos

        running_loss += loss.item()

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(trainloader):.4f}"
    )

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")